# Mapeamento de relatorios XLSX

Este notebook serve como ponto de partida para mapear relatorios em `.xlsx`, entender abas e colunas, padronizar nomes de campos e gerar uma saida tratada para uso posterior no Sangue Doce.

## 1. Configuracao inicial

Coloque os arquivos originais em `../data/raw/`. As saidas tratadas podem ir para `../data/processed/` ou `../outputs/`.

In [36]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

PROJECT_DIR = Path.cwd().resolve().parent / "ia-sangue-doce"
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

PROJECT_DIR, RAW_DIR, PROCESSED_DIR, OUTPUTS_DIR

(PosixPath('/home/jander-nery/Projetos/sangue-doce-new/ia-sangue-doce'),
 PosixPath('/home/jander-nery/Projetos/sangue-doce-new/ia-sangue-doce/data/raw'),
 PosixPath('/home/jander-nery/Projetos/sangue-doce-new/ia-sangue-doce/data/processed'),
 PosixPath('/home/jander-nery/Projetos/sangue-doce-new/ia-sangue-doce/outputs'))

## 2. Localizar relatorios disponiveis

In [37]:
xls_files = sorted(RAW_DIR.glob("*.xls"))

if not xls_files:
    print(f"Nenhum XLS encontrado em: {RAW_DIR}")
else:
    for index, file_path in enumerate(xls_files, start=1):
        print(f"{index}. {file_path.name}")

1. SiSensingCGM-AA25087LYB-01.21.00.00.xls


Escolha o arquivo que sera analisado. Por padrao, usamos o primeiro arquivo encontrado.

In [38]:
report_path = xls_files[0] if xls_files else None
report_path

PosixPath('/home/jander-nery/Projetos/sangue-doce-new/ia-sangue-doce/data/raw/SiSensingCGM-AA25087LYB-01.21.00.00.xls')

## 3. Inspecionar abas

In [39]:
if report_path:
    workbook = pd.ExcelFile(report_path)
    workbook.sheet_names
else:
    workbook = None
    []

/home/jander-nery/Projetos/sangue-doce-new/ia-sangue-doce/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Defina abaixo a aba principal. Se o relatorio tiver mais de uma aba util, repita o processo para cada uma.

In [40]:
sheet_name = workbook.sheet_names[0] if workbook else None
sheet_name

'Sensor de glucose'

## 4. Carregar amostra

Se o cabecalho nao estiver na primeira linha, ajuste `header_row`.

In [41]:
header_row = 0

df = (
    pd.read_excel(report_path, sheet_name=sheet_name, header=header_row)
    if report_path and sheet_name
    else pd.DataFrame()
)

df.head(10)

,Hora,Leitura de sensor(mg/dL)
0,03-09-2026 14:02 GMT-4,153
1,03-09-2026 14:07 GMT-4,147
2,03-09-2026 14:12 GMT-4,144
3,03-09-2026 14:17 GMT-4,136
4,03-09-2026 14:22 GMT-4,136
5,03-09-2026 14:27 GMT-4,133
6,03-09-2026 14:32 GMT-4,127
7,03-09-2026 14:37 GMT-4,118
8,03-09-2026 14:42 GMT-4,126
9,03-09-2026 14:47 GMT-4,131


## 5. Diagnostico das colunas

In [42]:
column_profile = pd.DataFrame(
    {
        "coluna_original": df.columns,
        "tipo": [df[column].dtype for column in df.columns],
        "valores_nulos": [df[column].isna().sum() for column in df.columns],
        "exemplo": [df[column].dropna().iloc[0] if df[column].dropna().shape[0] else None for column in df.columns],
    }
)

column_profile

,coluna_original,tipo,valores_nulos,exemplo
0,Hora,object,0,03-09-2026 14:02 GMT-4
1,Leitura de sensor(mg/dL),int64,0,153


## 6. Definir mapeamento

Use este dicionario para converter os nomes do relatorio para nomes padronizados. Ajuste conforme o arquivo real.

In [43]:
COLUMN_MAPPING = {
    "Hora": "measured_at",
    "Leitura de sensor(mg/dL)": "glucose_value_mg_dl",
}

mapped_df = df.rename(columns=COLUMN_MAPPING).copy()
mapped_df.head()

,measured_at,glucose_value_mg_dl
0,03-09-2026 14:02 GMT-4,153
1,03-09-2026 14:07 GMT-4,147
2,03-09-2026 14:12 GMT-4,144
3,03-09-2026 14:17 GMT-4,136
4,03-09-2026 14:22 GMT-4,136


## 7. Normalizar dados

Inclua aqui regras de conversao de datas, numeros e textos. Mantenha uma regra por campo para facilitar manutencao.

In [45]:
mapped_df["measured_at"] = (
    mapped_df["measured_at"]
    .astype(str)
    .str.replace(r"\s+GMT[+-]\d{1,2}$", "", regex=True)
)

mapped_df["measured_at"] = pd.to_datetime(
    mapped_df["measured_at"],
    format="%d-%m-%Y %H:%M",
    errors="coerce",
)

mapped_df["measured_at"] = mapped_df["measured_at"].dt.strftime(
    "%Y-%m-%d %H:%M:%S.000"
)

mapped_df.info()
mapped_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1104 entries, 0 to 1103
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   measured_at          1104 non-null   object
 1   glucose_value_mg_dl  1104 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 17.4+ KB


,measured_at,glucose_value_mg_dl
0,2026-09-03 14:02:00.000,153
1,2026-09-03 14:07:00.000,147
2,2026-09-03 14:12:00.000,144
3,2026-09-03 14:17:00.000,136
4,2026-09-03 14:22:00.000,136


## 8. Validacoes simples

In [48]:
expected_columns = [
    "measured_at",
    "glucose_value_mg_dl",
]

missing_columns = [
    column for column in expected_columns
    if column not in mapped_df.columns
]

missing_columns

[]

## 9. Exportar resultado

In [49]:
if report_path:
    output_path = PROCESSED_DIR / f"{report_path.stem}-mapeado.csv"
    mapped_df.to_csv(output_path, index=False)
    print(f"Arquivo exportado: {output_path}")
else:
    print("Adicione um XLSX em data/raw antes de exportar.")

Arquivo exportado: /home/jander-nery/Projetos/sangue-doce-new/ia-sangue-doce/data/processed/SiSensingCGM-AA25087LYB-01.21.00.00-mapeado.csv


In [50]:
output_path = PROCESSED_DIR / "glicose_tratada.json"

mapped_df.to_json(
    output_path,
    orient="records",
    force_ascii=False,
    indent=2,
)

output_path

PosixPath('/home/jander-nery/Projetos/sangue-doce-new/ia-sangue-doce/data/processed/glicose_tratada.json')